# 03 - Silver Pedidos - Bike Store Medallion
Notebook de tratamento da camada Silver focado em pedidos.

Fontes utilizadas da camada Bronze:
- `bike_store.bronze.orders`
- `bike_store.bronze.order_items`
- `bike_store.bronze.staffs`
- `bike_store.bronze.stores`

## Transformações aplicadas
- Mapeamento de status: 1=Pending, 2=Processing, 3=Shipped, 4=Delivered
- Cálculo do valor total da venda (list_price * quantity - discount)
- Enriquecimento com nome do vendedor e loja

In [0]:
# Criando os dataframes de pedidos, itens dos pedidos, pessoal e lojas
df_orders      = spark.read.table("bike_store.bronze.orders")
df_order_items = spark.read.table("bike_store.bronze.order_items")
df_staffs      = spark.read.table("bike_store.bronze.staffs")
df_stores      = spark.read.table("bike_store.bronze.stores")

In [0]:
# Vistoriando os dados de pedidos
display(df_orders.head(10))

In [0]:
# Verificando os tipos de dados de pedidos
df_orders.printSchema()

In [0]:
# Vistoriando os dados de itens dos pedidos
display(df_order_items.head(10))

In [0]:
# Verificando os tipos de dados de pedidos
df_order_items.printSchema()

In [0]:
%run ./Utils/Qualidade_Dados

In [0]:
# Verificando ocorrência de registros nulos por colunas
nulos_por_coluna(df_orders, "orders")
nulos_por_coluna(df_order_items, "order_items")
nulos_por_coluna(df_staffs, "staffs")
nulos_por_coluna(df_stores, "stores")

## Join Orders + Staff + Stores

In [0]:
from pyspark.sql.functions import when, col
# Fazendo o merge dos dataframes e aplicando a regra de negócio do status do pedido
df_orders_enriched = df_orders \
    .join(df_staffs.select("staff_id", "first_name", "last_name"), on="staff_id", how="left") \
    .join(df_stores.select("store_id", "store_name"), on="store_id", how="left") \
    .withColumn("order_status",
        when(col("order_status") == 1, "Pending")
        .when(col("order_status") == 2, "Processing")
        .when(col("order_status") == 3, "Shipped")
        .when(col("order_status") == 4, "Delivered")
        .otherwise("Unknown")
    ) \
    .drop("staff_id", "store_id")

display(df_orders_enriched)

In [0]:
from pyspark.sql.functions import round as rnd

df_order_items_enriched = df_order_items \
    .withColumn("total_value",
        rnd(col("quantity") * col("list_price") * (1 - col("discount")), 2)
    )

display(df_order_items_enriched)

## Join Order_Itens

In [0]:
df_silver_order_items = df_order_items_enriched \
    .join(df_orders_enriched, on="order_id", how="left") \
    .select(
        col("order_id"),
        col("item_id"),
        col("product_id"),
        col("order_status"),
        col("order_date"),
        col("store_name"),
        col("first_name").alias("staff_first_name"),
        col("last_name").alias("staff_last_name"),
        col("quantity"),
        col("list_price"),
        col("discount"),
        col("total_value")
    )

display(df_silver_order_items)

In [0]:
# Concatenando nome do vendedor
from pyspark.sql.functions import concat, lit

df_silver_order_items = df_silver_order_items \
    .withColumn("staff_name", concat(col("staff_first_name"), lit(" "), col("staff_last_name"))) \
    .drop("staff_first_name", "staff_last_name")

display(df_silver_order_items)

In [0]:
# Salvando tabela de Pedidos na camada Silver
df_silver_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bike_store.silver.orders")

print("✅ bike_store.silver.orders salvo com sucesso!")